# 🇰🇭 Cambodia Food Price Prediction
### AI Course Final Project 

This notebook trains a machine learning model to predict food commodity prices in Cambodia,
using the **WFP (World Food Programme) Food Prices dataset** (2003–2026), covering 50 commodities
across 86 markets in 25 provinces.

**Pipeline:**
1. Load & explore the dataset
2. Clean & filter to reliable, recent data
3. Engineer time-series features (lags, rolling averages, seasonality)
4. Train baseline vs. ML models (Random Forest, XGBoost)
5. Evaluate against a naive "no-change" baseline
6. Save the trained model for deployment (e.g. a Telegram bot)

> Run all cells top to bottom. In Colab: **Runtime → Run all**


## 1. Setup — Install & Import Libraries

In [ ]:
!pip install -q xgboost joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42


## 2. Load the Dataset

Upload `wfp_food_prices_khm.csv` when prompted (Colab file picker), **or** mount Google Drive
and point `DATA_PATH` to the file if you've saved it there instead.


In [ ]:
# --- Option A: Upload directly (recommended for a quick run) ---
from google.colab import files
uploaded = files.upload()  # select wfp_food_prices_khm.csv when prompted
DATA_PATH = list(uploaded.keys())[0]

# --- Option B: Use Google Drive instead (uncomment if you prefer this) ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/wfp_food_prices_khm.csv'


In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
print("Shape:", df.shape)
df.head()


## 3. Exploratory Data Analysis (EDA)

Before modeling, we need to understand:
- How far back the data goes and where coverage is strongest
- Which commodities/markets have enough history to model reliably
- Whether the target variable needs cleaning


In [ ]:
print("Date range:", df['date'].min().date(), "to", df['date'].max().date())
print("Missing values per column:\n", df.isnull().sum())
print("\nUnique commodities:", df['commodity'].nunique())
print("Unique markets:", df['market'].nunique())
print("Unique provinces (admin1):", df['admin1'].nunique())


In [ ]:
# Records per year — reveals reporting coverage over time
yearly_counts = df.groupby(df['date'].dt.year).size()
yearly_counts.plot(kind='bar', color='#34d399')
plt.title("Records per Year")
plt.xlabel("Year")
plt.ylabel("Number of price observations")
plt.tight_layout()
plt.show()


**Observation:** Reporting coverage before ~2019 is sparse (under 1,000 records/year).
From 2019 onward, coverage jumps dramatically (9,000–18,000 records/year) as more markets and
commodities were added. We'll train on **2019 onward** so the model learns from consistent,
comparable data — mixing in the sparse early years would hurt lag-feature quality and bias
the model toward provinces that only have old data.


In [ ]:
# Category breakdown
plt.figure(figsize=(8,5))
df['category'].value_counts().plot(kind='barh', color='#34d399')
plt.title("Records by Food Category")
plt.xlabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
# Price trend for a few staple commodities (using USD price for comparability)
staples = ['Rice (mixed, low quality)', 'Meat (pork, with fat)', 'Oil (vegetable)', 'Eggs (duck)']
fig, ax = plt.subplots(figsize=(12,6))
for c in staples:
    sub = df[df['commodity'] == c].groupby('date')['usdprice'].mean()
    ax.plot(sub.index, sub.values, label=c)
ax.set_title("Average USD Price Over Time — Selected Staples")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend()
plt.tight_layout()
plt.show()


## 4. Data Cleaning & Filtering

**Decisions made here (and why):**
- **Target variable = `usdprice`** — using USD instead of KHR avoids any local-currency
  quirks and makes the model's output directly interpretable/comparable across commodities.
- **Filter to 2019 onward** — as shown above, this is where coverage becomes dense and reliable.
- **Keep only commodities with ≥ 300 observations since 2019** — commodities with too few
  data points can't support reliable lag/rolling features and would just add noise.


In [ ]:
df_clean = df[df['date'] >= '2019-01-01'].copy()

commodity_counts = df_clean['commodity'].value_counts()
valid_commodities = commodity_counts[commodity_counts >= 300].index
df_clean = df_clean[df_clean['commodity'].isin(valid_commodities)]

print(f"Kept {df_clean['commodity'].nunique()} commodities out of {df['commodity'].nunique()}")
print(f"Rows after filtering: {len(df_clean)} (from {len(df)})")


In [ ]:
# Sort chronologically within each (commodity, market, pricetype) series —
# essential before building lag/rolling features
df_clean = df_clean.sort_values(['commodity', 'market', 'pricetype', 'date']).reset_index(drop=True)
df_clean.head()


## 5. Feature Engineering

We frame this as: *"given a commodity's recent price history at a specific market, predict
its next reported price."* This is more tractable for a course project than a full per-series
time-series model, and it generalizes to a Telegram bot query like `/predict rice phnom_penh`.

**Features:**
- `commodity`, `market`, `admin1` (province), `category`, `pricetype` — encoded as integers
- `month`, `quarter` — captures seasonality (harvest cycles, holidays, etc.)
- `lag_1` — price at the previous reporting period for this exact commodity+market+pricetype
- `lag_3` — price 3 periods ago
- `rolling_mean_3` — average of the 3 prior periods (smooths out noise/outliers)

**Target:** `usdprice` (the price we're predicting)


In [ ]:
group_cols = ['commodity', 'market', 'pricetype']

df_feat = df_clean.copy()
grp = df_feat.groupby(group_cols)['usdprice']

df_feat['lag_1'] = grp.shift(1)
df_feat['lag_3'] = grp.shift(3)
df_feat['rolling_mean_3'] = df_feat.groupby(group_cols)['usdprice'].transform(lambda s: s.shift(1).rolling(3).mean())

df_feat['month'] = df_feat['date'].dt.month
df_feat['quarter'] = df_feat['date'].dt.quarter
df_feat['year'] = df_feat['date'].dt.year

# Drop rows where lag features couldn't be computed (start of each series)
df_feat = df_feat.dropna(subset=['lag_1', 'lag_3', 'rolling_mean_3']).reset_index(drop=True)
print("Rows after adding lag features:", len(df_feat))
df_feat.head()


In [ ]:
# Encode categorical columns. We save the encoders — the Telegram bot will need
# the SAME mapping to turn a user's text input (e.g. "rice") into the right code.
categorical_cols = ['commodity', 'market', 'admin1', 'category', 'pricetype']
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_feat[col + '_enc'] = le.fit_transform(df_feat[col])
    encoders[col] = le

feature_cols = [c + '_enc' for c in categorical_cols] + ['month', 'quarter', 'lag_1', 'lag_3', 'rolling_mean_3']
target_col = 'usdprice'

X = df_feat[feature_cols]
y = df_feat[target_col]
print("Feature columns:", feature_cols)
print("X shape:", X.shape)


## 6. Train/Test Split — Time-Based

We split by **date**, not randomly. Randomly shuffling would let the model "see the future"
via nearby rows in the same series (data leakage) and make test accuracy look artificially
good. Training on the past and testing on a later, held-out period mimics how the model would
actually be used: predicting prices it hasn't seen yet.


In [ ]:
split_date = df_feat['date'].quantile(0.85)  # ~last 15% of the timeline held out for testing
print("Split date:", split_date)

train_mask = df_feat['date'] <= split_date
test_mask = df_feat['date'] > split_date

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")


## 7. Baseline Model — "Naive: Price Stays the Same"

Before trusting any ML model, we need a sanity-check baseline: if we simply predicted
*"next price = last known price"* (`lag_1`), how good would that be? Our ML model should
clearly beat this, or it isn't adding real value.


In [ ]:
naive_pred = X_test['lag_1']

baseline_mae = mean_absolute_error(y_test, naive_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, naive_pred))
baseline_r2 = r2_score(y_test, naive_pred)

print(f"Naive Baseline — MAE: {baseline_mae:.4f} | RMSE: {baseline_rmse:.4f} | R2: {baseline_r2:.4f}")


## 8. Train Models

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=3,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print(f"Random Forest — MAE: {rf_mae:.4f} | RMSE: {rf_rmse:.4f} | R2: {rf_r2:.4f}")


In [ ]:
xgb_model = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_r2 = r2_score(y_test, xgb_pred)

print(f"XGBoost — MAE: {xgb_mae:.4f} | RMSE: {xgb_rmse:.4f} | R2: {xgb_r2:.4f}")


## 9. Compare All Models

In [ ]:
results = pd.DataFrame({
    'Model': ['Naive Baseline', 'Random Forest', 'XGBoost'],
    'MAE (USD)': [baseline_mae, rf_mae, xgb_mae],
    'RMSE (USD)': [baseline_rmse, rf_rmse, xgb_rmse],
    'R2': [baseline_r2, rf_r2, xgb_r2],
})
results['MAE Improvement vs Baseline'] = (1 - results['MAE (USD)'] / baseline_mae) * 100
results


In [ ]:
results.set_index('Model')[['MAE (USD)', 'RMSE (USD)']].plot(kind='bar', color=['#34d399', '#059669'])
plt.title("Model Comparison — Lower is Better")
plt.ylabel("Error (USD)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 10. Feature Importance

Which signals matter most for predicting the next price? (Using the better-performing model
between Random Forest and XGBoost, picked automatically below.)


In [ ]:
best_model, best_name = (xgb_model, "XGBoost") if xgb_mae < rf_mae else (rf_model, "Random Forest")
print(f"Best model: {best_name}")

importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.plot(kind='barh', color='#34d399')
plt.title(f"Feature Importance — {best_name}")
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 11. Save the Model for Deployment

We save the trained model **and** the label encoders together — the encoders are required
so a future FastAPI/Telegram bot can convert a text input like `"rice"` / `"Phnom Penh"` into
the exact numeric codes the model expects.


In [ ]:
import os
os.makedirs('model_artifacts', exist_ok=True)

joblib.dump(best_model, 'model_artifacts/price_model.pkl')
joblib.dump(encoders, 'model_artifacts/encoders.pkl')
joblib.dump(feature_cols, 'model_artifacts/feature_cols.pkl')

print("Saved:")
print(" - model_artifacts/price_model.pkl")
print(" - model_artifacts/encoders.pkl")
print(" - model_artifacts/feature_cols.pkl")


In [ ]:
# Zip and download the artifacts folder
!zip -r model_artifacts.zip model_artifacts
from google.colab import files
files.download('model_artifacts.zip')


## 12. Example: Making a Prediction

This mirrors exactly what a Telegram bot backend would do: take a commodity + market name,
look up the most recent known prices for that series, build the feature row, and predict.


In [ ]:
from datetime import date

def next_calendar_period(today=None):
    """The month we forecast: the one after `today` (default: the real current date).

    Deliberately tied to the calendar rather than to where a series' data stops. A user
    asking in August wants to hear about September, not about the month after whatever the
    last WFP report happened to contain.
    """
    today = today or date.today()
    month = (today.month % 12) + 1
    year = today.year + 1 if month == 1 else today.year
    return year, month


def predict_next_price(commodity_name, market_name, pricetype_name='Retail', today=None):
    """Predict the price for next calendar month at a given commodity + market.

    month/quarter describe the month being forecast. The three lag features describe the most
    recent prices on record for that series. When a series is current those are adjacent and
    this is an ordinary one-step-ahead forecast; when a series stopped reporting years ago they
    are not, so the gap is printed rather than hidden.
    """
    subset = df_feat[
        (df_feat['commodity'] == commodity_name) &
        (df_feat['market'] == market_name) &
        (df_feat['pricetype'] == pricetype_name)
    ].sort_values('date')

    if subset.empty:
        return f"No historical data found for {commodity_name} in {market_name} ({pricetype_name})."

    last_row = subset.iloc[-1]
    target_year, target_month = next_calendar_period(today)
    target_quarter = ((target_month - 1) // 3) + 1

    row = pd.DataFrame([{
        'commodity_enc': encoders['commodity'].transform([commodity_name])[0],
        'market_enc': encoders['market'].transform([market_name])[0],
        'admin1_enc': encoders['admin1'].transform([last_row['admin1']])[0],
        'category_enc': encoders['category'].transform([last_row['category']])[0],
        'pricetype_enc': encoders['pricetype'].transform([pricetype_name])[0],
        'month': target_month,
        'quarter': target_quarter,
        'lag_1': last_row['usdprice'],
        'lag_3': subset.iloc[-3]['usdprice'] if len(subset) >= 3 else last_row['usdprice'],
        'rolling_mean_3': subset['usdprice'].tail(3).mean(),
    }])[feature_cols]

    predicted_price = best_model.predict(row)[0]
    last_price = last_row['usdprice']
    trend = "up" if predicted_price > last_price else "down"

    months_ahead = ((target_year - last_row['date'].year) * 12
                    + (target_month - last_row['date'].month))

    out = (f"{commodity_name} in {market_name} ({pricetype_name}):\n"
           f"  Last recorded ({last_row['date']:%b %Y}): ${last_price:.2f}\n"
           f"  Forecast for {target_year}-{target_month:02d}: ${predicted_price:.2f} ({trend})")

    if months_ahead > 1:
        out += (f"\n  NOTE: this series stopped reporting {months_ahead} months before the month\n"
                f"        forecast, so the lag features are that old. Treat it as an illustration\n"
                f"        of the method, not a current market estimate.")
    return out


# Example usage — try swapping in other commodity/market names from df_feat
print(predict_next_price('Rice (mixed, low quality)', 'Phnom Penh'))
print()
print(predict_next_price('Rice (mixed, low quality)', 'Phnom Penh', 'Wholesale'))


## Next Steps

- **Deploy:** wrap `predict_next_price()` in a FastAPI endpoint, load `model_artifacts/*.pkl` on startup.
- **Telegram bot:** use `python-telegram-bot` — a `/predict <commodity> <market>` command calls the API.
- **Report writeup:** include the model comparison table (Section 9) to show your model beats
  the naive baseline — that comparison is what reviewers/graders look for.
- **Possible extensions:** try a per-commodity time-series model (Prophet/ARIMA) as a comparison
  point, or add exchange-rate / rainfall data as extra features.
